# Esegui Analisi e Inferenza su Google Colab (via GitHub)
Questo notebook scarica automaticamente il tuo codice più aggiornato da GitHub.
Eseguirà `analyze.py` per fare inferenza sul test set, calcolare Dice e IoU, e generare le immagini comparative.

### 1. Scarica / Aggiorna il codice da GitHub
Questa cella clona la repository se non esiste ancora nella macchina virtuale di Colab, oppure fa un `git pull` per scaricare le ultime modifiche se l'hai già clonata in precedenza.

In [ ]:
import os

repo_url = "https://github.com/giorgio-di-dio/vessel-project.git"
repo_name = "vessel-project"

if not os.path.exists(repo_name):
    print(f"Clonazione della repository...")
    # L'opzione -b clona direttamente il branch specifico
    !git clone -b main {repo_url}
else:
    print("Repository già presente.")

# Spostati nella cartella del progetto
%cd {repo_name}

# Se la cartella esisteva già, scarica le novità e spostati sul branch corretto
!git fetch origin

!git pull origin main

### 2. Installa le librerie necessarie
Installa le dipendenze dal file `requirements.txt`.

In [ ]:
!pip install -r requirements.txt

### 3. Collega Google Drive
Colleghiamo Google Drive per due motivi:
1. **Recuperare il modello addestrato** (i pesi `.pth`) che hai salvato in precedenza.
2. **Salvare le immagini risultanti** in modo da non perderle quando la macchina virtuale di Colab verrà spenta.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### 4. Recupera il modello addestrato più recente da Drive
Questa cella cerca l'ultimo salvataggio fatto nella cartella `output_pesi` su Drive e copia il file `best_unet_vessel.pth` nella cartella locale `output/models/` in modo che `analyze.py` possa caricarlo.

In [ ]:
import os
import glob
import shutil

drive_output_base = "/content/drive/MyDrive/Advanced_Machine_Learning/output_pesi/*"
all_folders = [f for f in glob.glob(drive_output_base) if os.path.isdir(f)]

if not all_folders:
    print("ERRORE: Nessuna cartella trovata dentro output_pesi su Google Drive.")
    print("Assicurati di aver prima completato il training ed esportato i pesi.")
else:
    all_folders_sorted = sorted(all_folders, reverse=True)
    latest_folder = all_folders_sorted[0]
    print(f"Cartella pesi più recente individuata su Drive: {os.path.basename(latest_folder)}")
    
    model_path_drive = os.path.join(latest_folder, "models", "best_unet_vessel.pth")
    local_model_dir = "output/models"
    local_model_path = os.path.join(local_model_dir, "best_unet_vessel.pth")
    
    if os.path.exists(model_path_drive):
        os.makedirs(local_model_dir, exist_ok=True)
        shutil.copy2(model_path_drive, local_model_path)
        print(f"Modello copiato con successo in: {local_model_path}")
    else:
        print(f"ERRORE: Il file {model_path_drive} non esiste.")

### 5. Avvia l'Analisi!
Ora eseguiamo il file `analyze.py`. Verranno calcolate le metriche finali (Dice e IoU) sul test set, e verranno generate le immagini comparative in `output/results/`.

In [ ]:
!python analyze.py

### 6. Backup dei risultati (da eseguire A FINE analisi)
Esegui questa cella quando `analyze.py` ha finito, così ti salvi le immagini risultanti sul tuo Drive.

In [ ]:
import os
import datetime
from zoneinfo import ZoneInfo

run_timestamp = datetime.datetime.now(tz=ZoneInfo('Europe/Rome')).strftime("%Y%m%d_%H%M")
print(f"Timestamp run analisi: {run_timestamp}")

# Costruisci il percorso di destinazione su Google Drive
destination_path = f"/content/drive/MyDrive/Advanced_Machine_Learning/output_risultati_analisi/{run_timestamp}"

# Crea la cartella su Google Drive
print(f"Creazione della cartella di destinazione: {destination_path}")
!mkdir -p "{destination_path}"

# Copia i contenuti della cartella 'output/results'
print(f"Copia dei risultati nella cartella: {destination_path}")
!cp -r output/results/* "{destination_path}"

print("Salvataggio su Drive completato!")

### 7. Mostra alcune immagini generate
Stampiamo a video un paio di risultati (se presenti).

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import glob

# Cerca le immagini salvate in output/results/
result_images = sorted(glob.glob("output/results/result_*.png"))

if result_images:
    # Mostriamo solo le prime 5 per non riempire troppo il notebook
    for img_path in result_images[:5]:
        print(f"Mostrando: {img_path}")
        img = mpimg.imread(img_path)
        plt.figure(figsize=(18, 6))
        plt.imshow(img)
        plt.axis('off')
        plt.show()
else:
    print("Nessuna immagine di risultato trovata in output/results/.")

### 8. Rilascia la GPU
Quando hai finito, scollega l'ambiente.

In [ ]:
# Questa riga scollega il runtime e rilascia la GPU istantaneamente
#from google.colab import runtime
#runtime.unassign()